# Named Entity Recognition (NER) Pipeline

## Project Information
- **Source**: Transformers Packt Course (lazyprogrammer.me/course_files/nlp/)
- **Objective**: Identify and classify named entities in text
- **Pipeline**: `ner` (Named Entity Recognition)
- **Model**: BERT-based models fine-tuned on NER tasks (e.g., CoNLL-2003)

## Overview
This notebook demonstrates how to use the NER pipeline to identify and classify named entities such as persons (PER), organizations (ORG), locations (LOC), and more in text.


In [ ]:
%pip install transformers


In [ ]:
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')


## Initialize NER Pipeline


In [ ]:
# Initialize the NER pipeline with aggregation strategy
# 'simple' strategy groups subword tokens into complete entities
ner = pipeline("ner", aggregation_strategy='simple')
print(f"Pipeline initialized: {type(ner)}")
print("Using default NER model (typically BERT fine-tuned on CoNLL-2003)")


## Basic NER Examples


In [ ]:
# Example 1: Simple sentence with various entity types
text = "Steve Jobs is the CEO of Apple, headquartered in California, USA."
entities = ner(text)

print(f"Text: {text}\n")
print("Detected Entities:")
print("-" * 60)
for entity in entities:
    print(f"Entity: {entity['word']:20s} | Type: {entity['entity_group']:10s} | Score: {entity['score']:.4f}")
    print(f"  Position: {entity['start']}-{entity['end']}")
    print()


## More Complex Examples


In [ ]:
# Example 2: News article snippet
text = """
Elon Musk, the CEO of Tesla and SpaceX, announced plans to build a new factory 
in Austin, Texas. The company will invest $1.1 billion in the project, creating 
thousands of jobs. The announcement was made during a press conference at the 
Tesla headquarters in Palo Alto, California.
"""

entities = ner(text)

print("Text:")
print(text)
print("\n" + "=" * 60)
print("Detected Entities:")
print("=" * 60)

# Group by entity type
entity_groups = {}
for entity in entities:
    entity_type = entity['entity_group']
    if entity_type not in entity_groups:
        entity_groups[entity_type] = []
    entity_groups[entity_type].append(entity)

# Display grouped entities
for entity_type, entity_list in entity_groups.items():
    print(f"\n{entity_type}:")
    for entity in entity_list:
        print(f"  - {entity['word']} (score: {entity['score']:.4f})")


## Entity Types

Common entity types recognized by NER models:


In [ ]:
# Test different entity types
examples = [
    ("PER - Person", "Barack Obama was the 44th President of the United States."),
    ("ORG - Organization", "Microsoft and Google are leading technology companies."),
    ("LOC - Location", "The conference will be held in Paris, France."),
    ("MISC - Miscellaneous", "The Super Bowl is one of the biggest sporting events."),
]

print("Entity Type Examples:\n")
print("=" * 80)

for label, text in examples:
    entities = ner(text)
    print(f"\n{label}")
    print(f"Text: {text}")
    print("Entities found:")
    for entity in entities:
        print(f"  - {entity['word']} ({entity['entity_group']}, score: {entity['score']:.4f})")
    print("-" * 80)


## Visualize Entities in Text


In [ ]:
def highlight_entities(text, entities):
    """
    Create a simple visualization of entities in text.
    """
    # Sort entities by start position
    sorted_entities = sorted(entities, key=lambda x: x['start'])
    
    result = []
    last_end = 0
    
    for entity in sorted_entities:
        # Add text before entity
        if entity['start'] > last_end:
            result.append(text[last_end:entity['start']])
        
        # Add highlighted entity
        entity_text = text[entity['start']:entity['end']]
        result.append(f"[{entity_text}]({entity['entity_group']})")
        last_end = entity['end']
    
    # Add remaining text
    if last_end < len(text):
        result.append(text[last_end:])
    
    return ''.join(result)

# Example
text = "Tim Cook is the CEO of Apple Inc., based in Cupertino, California."
entities = ner(text)

print("Original text:")
print(text)
print("\nEntities with types:")
for entity in entities:
    print(f"  {entity['word']} -> {entity['entity_group']} (confidence: {entity['score']:.4f})")

print("\nHighlighted text:")
highlighted = highlight_entities(text, entities)
print(highlighted)
